# Lab 05 — Lakeflow Declarative Pipelines
## 02 — Final Validation

This notebook validates the completed Lab 05 pipeline **after the Lakeflow pipeline has run**.

It does not create or modify pipeline datasets.

### Validation goals
- Confirm all Bronze, Silver, and Gold datasets exist
- Verify source-file ingestion completeness
- Verify Silver expectations left no invalid production rows
- Confirm the `station_id` enrichment join is successful
- Validate Gold output
- Check replay/incremental-ingestion invariants
- Produce one final PASS/FAIL summary

### Important
Expectation pass/fail metrics and the declarative lineage graph should still be captured from the **Lakeflow pipeline UI**. This notebook validates the resulting data, while the UI provides the pipeline-specific expectation and lineage evidence.


## 1. Runtime parameters

The validation notebook uses the catalog and schema in which the Lakeflow pipeline publishes its datasets.

`run_checks=true` makes failed invariants raise an exception at the end. Set it to `false` only when you want to inspect results without failing the notebook.


In [0]:
dbutils.widgets.text("catalog", "dbr_dev", "Catalog")
dbutils.widgets.text("schema", "parvinbadalov", "Schema")
dbutils.widgets.text("volume_name", "lab05_lakeflow", "Managed volume")
dbutils.widgets.text(
    "streaming_volume_name",
    "lab05_lakeflow_streaming",
    "Streaming external volume"
)
dbutils.widgets.dropdown(
    "run_checks",
    "true",
    ["true", "false"],
    "Run checks"
)

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
volume_name = dbutils.widgets.get("volume_name").strip()
streaming_volume_name = dbutils.widgets.get(
    "streaming_volume_name"
).strip()
run_checks = (
    dbutils.widgets.get("run_checks").strip().lower()
    == "true"
)

volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"
streaming_volume_path = (
    f"/Volumes/{catalog}/{schema}/{streaming_volume_name}"
)
status_landing_path = (
    f"{streaming_volume_path}/landing/station_status"
)

print(f"Catalog          : {catalog}")
print(f"Schema           : {schema}")
print(f"Managed volume   : {volume_name}")
print(f"Streaming volume : {streaming_volume_name}")
print(f"Run checks       : {run_checks}")


Catalog          : dbr_dev
Schema           : parvinbadalov
Managed volume   : lab05_lakeflow
Streaming volume : lab05_lakeflow_streaming
Run checks       : True


## 2. Expected pipeline datasets

The final Lab 05 declarative graph should publish:

### Bronze
- `station_status_bronze`
- `station_information_bronze`

### Silver
- `station_status_silver`
- `station_information_silver`
- `station_status_enriched_silver`

### Gold
- `station_summary_gold`

The setup notebook does **not** create these objects. Their existence proves that Lakeflow evaluated the declarative source definitions.


In [0]:
TABLES = {
    "station_status_bronze":
        f"{catalog}.{schema}.station_status_bronze",
    "station_information_bronze":
        f"{catalog}.{schema}.station_information_bronze",
    "station_status_silver":
        f"{catalog}.{schema}.station_status_silver",
    "station_information_silver":
        f"{catalog}.{schema}.station_information_silver",
    "station_status_enriched_silver":
        f"{catalog}.{schema}.station_status_enriched_silver",
    "station_summary_gold":
        f"{catalog}.{schema}.station_summary_gold",
}

existence_results = []

for logical_name, full_name in TABLES.items():
    exists = spark.catalog.tableExists(full_name)
    existence_results.append(
        (
            logical_name,
            full_name,
            "PASS" if exists else "FAIL",
        )
    )

existence_df = spark.createDataFrame(
    existence_results,
    ["dataset", "full_name", "result"],
)

display(existence_df)


dataset,full_name,result
station_status_bronze,dbr_dev.parvinbadalov.station_status_bronze,PASS
station_information_bronze,dbr_dev.parvinbadalov.station_information_bronze,PASS
station_status_silver,dbr_dev.parvinbadalov.station_status_silver,PASS
station_information_silver,dbr_dev.parvinbadalov.station_information_silver,PASS
station_status_enriched_silver,dbr_dev.parvinbadalov.station_status_enriched_silver,PASS
station_summary_gold,dbr_dev.parvinbadalov.station_summary_gold,PASS


In [0]:
missing_tables = [
    full_name
    for _, full_name, result in existence_results
    if result == "FAIL"
]

if missing_tables:
    raise RuntimeError(
        "Lakeflow pipeline datasets are missing. "
        "Run the Lab 05 Lakeflow pipeline first. Missing: "
        + ", ".join(missing_tables)
    )

print("✅ All expected Lab 05 datasets exist.")


✅ All expected Lab 05 datasets exist.


## 3. Load the final datasets

All remaining checks are read-only.

Using fully qualified table names makes the validation notebook independent of the notebook's current catalog/schema selection.


In [0]:
status_bronze_df = spark.table(
    TABLES["station_status_bronze"]
)

information_bronze_df = spark.table(
    TABLES["station_information_bronze"]
)

status_silver_df = spark.table(
    TABLES["station_status_silver"]
)

information_silver_df = spark.table(
    TABLES["station_information_silver"]
)

enriched_silver_df = spark.table(
    TABLES["station_status_enriched_silver"]
)

gold_df = spark.table(
    TABLES["station_summary_gold"]
)

print("✅ Final datasets loaded for validation.")


✅ Final datasets loaded for validation.


## 4. Row-count overview

This gives a compact view of the medallion layers.

Expected pattern:
- `station_status_bronze`: one row per raw status snapshot document
- `station_information_bronze`: one raw reference document
- `station_status_silver`: one row per station per snapshot
- `station_information_silver`: one row per current station
- `station_status_enriched_silver`: validated status observations enriched with reference data
- `station_summary_gold`: one analytical row per station/reference grouping


In [0]:
dataset_counts = [
    ("station_status_bronze", status_bronze_df.count()),
    (
        "station_information_bronze",
        information_bronze_df.count(),
    ),
    ("station_status_silver", status_silver_df.count()),
    (
        "station_information_silver",
        information_silver_df.count(),
    ),
    (
        "station_status_enriched_silver",
        enriched_silver_df.count(),
    ),
    ("station_summary_gold", gold_df.count()),
]

display(
    spark.createDataFrame(
        dataset_counts,
        ["dataset", "row_count"],
    )
)


dataset,row_count
station_status_bronze,12
station_information_bronze,1
station_status_silver,30108
station_information_silver,2509
station_status_enriched_silver,30108
station_summary_gold,2509


## 5. Streaming-source ingestion completeness

Every producer execution creates one immutable JSON file.

Bronze is intentionally modeled as **one raw GBFS document per source file**, so the strongest simple ingestion invariant is:

```text
number of station_status JSON files
        =
Bronze raw documents
        =
distinct Bronze source files
```

If this remains true after a rerun with no new files, the pipeline has not duplicated previously processed files.

After adding one new producer snapshot and refreshing the pipeline, both sides should increase by one.


In [0]:
from pyspark.sql import functions as F


status_source_files = [
    f
    for f in dbutils.fs.ls(status_landing_path)
    if f.name.endswith(".json")
]

status_source_file_count = len(status_source_files)

status_bronze_count = status_bronze_df.count()

status_bronze_distinct_files = (
    status_bronze_df
    .select("_source_file")
    .distinct()
    .count()
)

ingestion_profile = [
    ("status_source_json_files", status_source_file_count),
    ("status_bronze_documents", status_bronze_count),
    (
        "status_bronze_distinct_source_files",
        status_bronze_distinct_files,
    ),
]

display(
    spark.createDataFrame(
        ingestion_profile,
        ["metric", "value"],
    )
)


metric,value
status_source_json_files,12
status_bronze_documents,12
status_bronze_distinct_source_files,12


## 6. Silver status quality invariants

The production `station_status_silver` dataset uses Lakeflow expectations.

After `expect_all_or_drop`, these invalid conditions should not remain:
- null `station_id`
- negative `num_bikes_available`
- negative `num_docks_available`

We also verify `status_record_id` uniqueness. It is built from:

```text
_source_file + station_id
```

so it represents one station observation within one source snapshot.


In [0]:
status_quality_metrics = {
    "null_station_id": (
        status_silver_df
        .filter(F.col("station_id").isNull())
        .count()
    ),
    "negative_bikes": (
        status_silver_df
        .filter(F.col("num_bikes_available") < 0)
        .count()
    ),
    "negative_docks": (
        status_silver_df
        .filter(F.col("num_docks_available") < 0)
        .count()
    ),
    "duplicate_status_record_id": (
        status_silver_df
        .groupBy("status_record_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    ),
}

display(
    spark.createDataFrame(
        [
            (metric, value)
            for metric, value
            in status_quality_metrics.items()
        ],
        ["metric", "invalid_row_count"],
    )
)


metric,invalid_row_count
null_station_id,0
negative_bikes,0
negative_docks,0
duplicate_status_record_id,0


## 7. Silver reference-data quality invariants

The `station_information_silver` expectations protect:
- `station_id`
- valid latitude range
- valid longitude range
- non-negative capacity

Null capacity is allowed by the production rule and monitored separately, but a negative capacity is invalid.


In [0]:
information_quality_metrics = {
    "null_station_id": (
        information_silver_df
        .filter(F.col("station_id").isNull())
        .count()
    ),
    "invalid_latitude": (
        information_silver_df
        .filter(
            F.col("latitude").isNotNull()
            & (
                (F.col("latitude") < -90)
                | (F.col("latitude") > 90)
            )
        )
        .count()
    ),
    "invalid_longitude": (
        information_silver_df
        .filter(
            F.col("longitude").isNotNull()
            & (
                (F.col("longitude") < -180)
                | (F.col("longitude") > 180)
            )
        )
        .count()
    ),
    "negative_capacity": (
        information_silver_df
        .filter(F.col("capacity") < 0)
        .count()
    ),
    "duplicate_station_id": (
        information_silver_df
        .groupBy("station_id")
        .count()
        .filter(
            F.col("station_id").isNotNull()
            & (F.col("count") > 1)
        )
        .count()
    ),
}

display(
    spark.createDataFrame(
        [
            (metric, value)
            for metric, value
            in information_quality_metrics.items()
        ],
        ["metric", "invalid_row_count"],
    )
)


metric,invalid_row_count
null_station_id,0
invalid_latitude,0
invalid_longitude,0
negative_capacity,0
duplicate_station_id,0


## 8. Validate Silver enrichment

`station_status_enriched_silver` is the stream-static enrichment join:

```text
station_status_silver
        +
station_information_silver
        |
        | station_id
        v
station_status_enriched_silver
```

The source-preparation notebook previously measured 100% join coverage. Here we validate the actual pipeline result.

The enrichment also calculates:
- `bike_availability_pct`
- `dock_availability_pct`
- `station_reference_matched`


In [0]:
enriched_total = enriched_silver_df.count()

enriched_matched = (
    enriched_silver_df
    .filter(F.col("station_reference_matched"))
    .count()
)

enriched_unmatched = enriched_total - enriched_matched

enriched_join_coverage_pct = (
    enriched_matched / enriched_total * 100
    if enriched_total
    else 0.0
)

enrichment_profile = [
    ("enriched_rows", str(enriched_total)),
    ("matched_rows", str(enriched_matched)),
    ("unmatched_rows", str(enriched_unmatched)),
    (
        "join_coverage_pct",
        f"{enriched_join_coverage_pct:.2f}%",
    ),
]

display(
    spark.createDataFrame(
        enrichment_profile,
        ["metric", "value"],
    )
)


metric,value
enriched_rows,30108
matched_rows,30108
unmatched_rows,0
join_coverage_pct,100.00%


## 9. Validate Gold output

The Gold materialized view should:
- contain rows
- have positive observation counts
- have one analytical summary per grouping
- retain no unmatched reference observations for the current clean source
- produce availability bands from the Silver metrics


In [0]:
gold_invalid_metrics = {
    "zero_or_negative_observation_count": (
        gold_df
        .filter(F.col("observation_count") <= 0)
        .count()
    ),
    "unmatched_reference_observations": (
        gold_df
        .filter(
            F.col("unmatched_reference_observations") > 0
        )
        .count()
    ),
    "null_availability_band": (
        gold_df
        .filter(F.col("availability_band").isNull())
        .count()
    ),
}

display(
    spark.createDataFrame(
        [
            (metric, value)
            for metric, value
            in gold_invalid_metrics.items()
        ],
        ["metric", "invalid_row_count"],
    )
)

display(
    gold_df
    .orderBy(
        F.desc("observation_count"),
        F.asc("station_name"),
    )
    .limit(20)
)


metric,invalid_row_count
zero_or_negative_observation_count,0
unmatched_reference_observations,0
null_availability_band,0


station_id,station_name,short_name,latitude,longitude,region_id,capacity,observation_count,first_observed_at,last_observed_at,avg_bikes_available,min_bikes_available,max_bikes_available,avg_docks_available,min_docks_available,max_docks_available,avg_bike_availability_pct,avg_dock_availability_pct,avg_ebikes_available,not_renting_observations,not_returning_observations,unmatched_reference_observations,avg_station_utilization_pct,availability_band
0b613329-b8c8-4b8c-ad40-79952aa41157,1 Ave & E 110 St,7522.02,40.7923272,-73.9383,71,25,12,2026-08-16T21:22:32.000Z,2026-08-16T22:29:18.000Z,11.33,11,12,9.67,9,10,45.33,38.67,0.33,0,0,0,45.32,BALANCED
1958457675193833900,1 Ave & E 118 St,7596.11,40.797459,-73.934499,71,24,12,2026-08-16T21:22:50.000Z,2026-08-16T22:28:38.000Z,8.75,6,9,12.25,12,15,36.46,51.04,2.0,0,0,0,36.46,BALANCED
66dc7802-0aca-11e7-82f6-3863bb44ef7c,1 Ave & E 16 St,5779.08,40.73221853,-73.98165557,71,54,12,2026-08-16T21:21:11.000Z,2026-08-16T22:29:49.000Z,3.08,2,4,48.08,47,49,5.71,89.04,2.08,0,0,0,5.7,VERY_LOW
66dc8f1f-0aca-11e7-82f6-3863bb44ef7c,1 Ave & E 18 St,5854.09,40.733812191966315,-73.98054420948029,71,54,12,2026-08-16T21:22:56.000Z,2026-08-16T22:29:42.000Z,6.75,2,8,42.08,41,45,12.5,77.93,2.75,0,0,0,12.5,LOW
66dc89ae-0aca-11e7-82f6-3863bb44ef7c,1 Ave & E 30 St,6079.03,40.74144387,-73.97536082,71,38,12,2026-08-16T21:22:58.000Z,2026-08-16T22:29:57.000Z,11.75,11,12,24.25,24,25,30.92,63.82,8.33,0,0,0,30.92,BALANCED
1917312704757770810,1 Ave & E 38 St,6230.02,40.746202,-73.971822,71,39,12,2026-08-16T21:22:26.000Z,2026-08-16T22:29:25.000Z,9.25,2,11,27.92,26,35,23.72,71.58,6.75,0,0,0,23.72,LOW
1984510483573713186,1 Ave & E 42 St,6303.02,40.7487575,-73.970127,71,62,12,2026-08-16T21:22:19.000Z,2026-08-16T22:29:43.000Z,26.42,26,31,28.83,27,29,42.61,46.5,11.08,0,0,0,42.61,BALANCED
66dc2172-0aca-11e7-82f6-3863bb44ef7c,1 Ave & E 44 St,6379.03,40.75001986,-73.96905301,71,87,12,2026-08-16T21:22:39.000Z,2026-08-16T22:28:56.000Z,9.0,9,9,74.0,74,74,10.34,85.06,0.08,0,0,0,10.34,LOW
c37931bb-8571-4671-a9a8-f3cf23897680,1 Ave & E 6 St,5626.15,40.72633085971696,-73.98616880178452,71,94,12,2026-08-16T21:22:53.000Z,2026-08-16T22:29:25.000Z,31.17,30,36,60.92,56,63,33.16,64.8,21.67,0,0,0,33.16,BALANCED
66dd4356-0aca-11e7-82f6-3863bb44ef7c,1 Ave & E 62 St,6753.08,40.7612274,-73.96094022,71,45,12,2026-08-16T21:22:50.000Z,2026-08-16T22:29:52.000Z,23.42,22,31,15.17,9,16,52.04,33.71,13.0,0,0,0,52.04,BALANCED


## 10. Final validation matrix

This section consolidates the important invariants into one PASS/FAIL table.

The ingestion-file equality is especially useful for the lab's **safe reload** requirement:

```text
source files == Bronze documents == distinct Bronze source files
```

It proves that the current set of files has been represented exactly once in Bronze.

For the incremental test:
1. Capture this validation result.
2. Run `citibike_status_producer.py` once.
3. Refresh the Lakeflow pipeline.
4. Rerun this notebook.
5. Verify the three ingestion counts all increase consistently.


In [0]:
checks = [
    (
        "all_expected_datasets_exist",
        len(missing_tables) == 0,
    ),
    (
        "status_source_files_match_bronze_documents",
        status_source_file_count == status_bronze_count,
    ),
    (
        "status_bronze_one_row_per_source_file",
        status_bronze_count
        == status_bronze_distinct_files,
    ),
    (
        "status_silver_has_rows",
        status_silver_df.count() > 0,
    ),
    (
        "information_silver_has_rows",
        information_silver_df.count() > 0,
    ),
    (
        "enriched_silver_has_rows",
        enriched_total > 0,
    ),
    (
        "gold_has_rows",
        gold_df.count() > 0,
    ),
    (
        "status_station_id_valid",
        status_quality_metrics["null_station_id"] == 0,
    ),
    (
        "status_bike_counts_valid",
        status_quality_metrics["negative_bikes"] == 0,
    ),
    (
        "status_dock_counts_valid",
        status_quality_metrics["negative_docks"] == 0,
    ),
    (
        "status_record_id_unique",
        status_quality_metrics[
            "duplicate_status_record_id"
        ] == 0,
    ),
    (
        "information_station_id_valid",
        information_quality_metrics[
            "null_station_id"
        ] == 0,
    ),
    (
        "information_latitude_valid",
        information_quality_metrics[
            "invalid_latitude"
        ] == 0,
    ),
    (
        "information_longitude_valid",
        information_quality_metrics[
            "invalid_longitude"
        ] == 0,
    ),
    (
        "information_capacity_valid",
        information_quality_metrics[
            "negative_capacity"
        ] == 0,
    ),
    (
        "information_station_id_unique",
        information_quality_metrics[
            "duplicate_station_id"
        ] == 0,
    ),
    (
        "station_reference_join_100_pct",
        enriched_unmatched == 0,
    ),
    (
        "gold_observation_counts_valid",
        gold_invalid_metrics[
            "zero_or_negative_observation_count"
        ] == 0,
    ),
    (
        "gold_reference_matches_valid",
        gold_invalid_metrics[
            "unmatched_reference_observations"
        ] == 0,
    ),
    (
        "gold_availability_band_present",
        gold_invalid_metrics[
            "null_availability_band"
        ] == 0,
    ),
]

validation_summary_df = spark.createDataFrame(
    [
        (
            check_name,
            "PASS" if passed else "FAIL",
        )
        for check_name, passed in checks
    ],
    ["check", "result"],
)

display(validation_summary_df)


check,result
all_expected_datasets_exist,PASS
status_source_files_match_bronze_documents,PASS
status_bronze_one_row_per_source_file,PASS
status_silver_has_rows,PASS
information_silver_has_rows,PASS
enriched_silver_has_rows,PASS
gold_has_rows,PASS
status_station_id_valid,PASS
status_bike_counts_valid,PASS
status_dock_counts_valid,PASS


In [0]:
failed_checks = [
    check_name
    for check_name, passed in checks
    if not passed
]

if failed_checks and run_checks:
    raise AssertionError(
        "Lab 05 validation failed: "
        + ", ".join(failed_checks)
    )

if failed_checks:
    print(
        "⚠️ LAB 05 VALIDATION COMPLETED WITH FAILURES"
    )
    print(
        "Failed checks: "
        + ", ".join(failed_checks)
    )
else:
    print("✅ LAB 05 VALIDATION PASSED")
    print()
    print(
        f"Status source files     : "
        f"{status_source_file_count:,}"
    )
    print(
        f"Bronze status documents : "
        f"{status_bronze_count:,}"
    )
    print(
        f"Silver status rows       : "
        f"{status_silver_df.count():,}"
    )
    print(
        f"Reference stations       : "
        f"{information_silver_df.count():,}"
    )
    print(
        f"Enriched rows            : "
        f"{enriched_total:,}"
    )
    print(
        f"Join coverage            : "
        f"{enriched_join_coverage_pct:.2f}%"
    )
    print(
        f"Gold station rows        : "
        f"{gold_df.count():,}"
    )


✅ LAB 05 VALIDATION PASSED

Status source files     : 12
Bronze status documents : 12
Silver status rows       : 30,108
Reference stations       : 2,509
Enriched rows            : 30,108
Join coverage            : 100.00%
Gold station rows        : 2,509


## 11. Completion evidence

Lab 05 functional validation is complete.

The following evidence has been captured from the Lakeflow pipeline and validation notebook:

- Pipeline completed successfully
- Declarative lineage / DAG is visible
- `station_status_silver` expectations passed
- `station_information_silver` expectations passed
- Safe rerun confirmed no duplicate ingestion
- Incremental ingestion confirmed one new source file was processed
- Final validation completed successfully
- All validation checks passed

Final validation result:

```text
Status source files      : 12
Bronze status documents  : 12
Silver status rows       : 30,108
Reference stations       : 2,509
Enriched rows            : 30,108
Join coverage            : 100.00%
Gold station rows        : 2,509